# 내 목소리로 대본 읽기 — AutoCAD 2024 실습 강의

Qwen3-TTS 로 선생님 목소리를 복제해 여덟 차시 낭독 대본을 읽힌다.
**끊겨도 이어서 한다.** 세션이 죽으면 이 노트북을 다시 열고 3번 셀부터 다시 돌리면 된다.

## 시작하기 전에 — 구글 드라이브에 이 세 개를 올려 둔다

    내 드라이브/autocad-narration/
      jobs-ko.json        저장소에서 만든 작업 목록
      speak_clone.py       저장소 scripts/part/speak_clone.py
      refs/base.wav       선생님 목소리 (10~15초 권장, 3초 이상)

`refs/refs.json` 은 4번 셀이 틀을 만들어 준다. 거기 **base.wav 에서 실제로 들리는 말을
글자까지 그대로** 적는다. 이 글이 음성과 다르면 복제 품질이 떨어진다.

## 런타임

메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU**.


## 1. 설치

`qwen-tts` 와 `soundfile`. 1~2분 걸린다.

> 이 컴퓨터에 GPU 가 있으면 Colab 없이 바로 돌려도 된다:
> `python scripts/part/speak_clone.py --work local-materials/qwen-narration --lang ko`

In [ ]:
!pip install -q -U qwen-tts soundfile

## 2. 드라이브 연결

구글 계정 권한을 묻는다. 결과가 드라이브에 남아야 세션이 끊겨도 이어서 할 수 있다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/autocad-narration'
LANG = 'ko'
print(WORK)

## 3. 폴더 확인

올려 둔 파일이 다 있는지 본다. 없는 게 있으면 여기서 알려 준다.

In [ ]:
import os, json
need = ['jobs-%s.json' % LANG, 'speak_clone.py']
for name in need:
    path = os.path.join(WORK, name)
    print(('있음  ' if os.path.exists(path) else '없음! ') + name)
refs = os.path.join(WORK, 'refs')
print('refs/ :', sorted(os.listdir(refs)) if os.path.isdir(refs) else '없음! 폴더를 만들고 목소리 wav 를 넣어라')

import torch
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음! 런타임 유형을 T4 GPU 로 바꿔라')

doc = json.load(open(os.path.join(WORK, 'jobs-%s.json' % LANG), encoding='utf-8'))
jobs = [j for l in doc['lessons'] for j in l['jobs']]
print('작업  : 차시 %d · 문단 %d · 토막 %d'
      % (len(doc['lessons']), len(jobs), sum(len(j['chunks']) for j in jobs)))

## 4. 읽기

처음 돌리면 `refs/refs.json` 틀만 만들고 멈춘다 — 채운 뒤 다시 돌린다.
채워져 있으면 남은 토막부터 만든다. **끊기면 이 셀만 다시 돌리면 된다.**

`--limit 50` 을 붙이면 50개만 만들어 본다. 한 차시만 하려면 `--lesson lesson-02`.

In [ ]:
!python {WORK}/speak_clone.py --work {WORK} --lang {LANG}

## 5. 어디까지 왔나

만든 토막 수, 남은 수, 길이가 수상한 토막을 본다.
수상한 토막은 `out/ko/suspect.jsonl` 에 원문과 함께 남아 있다 — 들어보고 판단한다.

In [ ]:
import os, json, collections
out = os.path.join(WORK, 'out', LANG)
doc = json.load(open(os.path.join(WORK, 'jobs-%s.json' % LANG), encoding='utf-8'))
total = sum(len(j['chunks']) for l in doc['lessons'] for j in l['jobs'])

done, seconds = {}, 0.0
path = os.path.join(out, 'done.jsonl')
if os.path.exists(path):
    for line in open(path, encoding='utf-8'):
        if line.strip():
            r = json.loads(line)
            done[(r['slug'], r['key'], r['n'])] = r
seconds = sum(r['sec'] for r in done.values())
print('토막 %d / %d (%.0f%%) · 소리 %.0f분' % (len(done), total, 100.0*len(done)/total, seconds/60))

per = collections.Counter(k[0] for k in done)
for l in doc['lessons']:
    want = sum(len(j['chunks']) for j in l['jobs'])
    print('  %-34s %4d / %4d' % (l['slug'], per.get(l['slug'], 0), want))

bad = os.path.join(out, 'suspect.jsonl')
rows = [json.loads(x) for x in open(bad, encoding='utf-8')] if os.path.exists(bad) else []
print('\n길이가 수상한 토막 %d개' % len(rows))
for r in rows[:10]:
    print('  %s#%02d  %.1f초 (기대 %.1f) · %s' % (r['key'], r['n'], r['sec'], r['want'], r['text'][:40]))

## 6. 내려받기

다 되면 하나로 묶는다. 이 zip 을 내 컴퓨터의 저장소로 가져가
`python scripts/part/ingest_voice.py` 에 넘긴다.

In [ ]:
import shutil
zippath = '/content/narration-%s' % LANG
shutil.make_archive(zippath, 'zip', os.path.join(WORK, 'out', LANG))
print('%s.zip  %.0f MB' % (zippath, os.path.getsize(zippath + '.zip') / 1e6))
from google.colab import files
files.download(zippath + '.zip')